# Phase 3: Analysis

Three patterns from Phase 2: Q4 spending surge, amendment growth, and vendor concentration. This notebook tests each with evidence and ends with recommendations.

**Scope**: National Defence excluded. Only valid `reporting_period` rows. Eras analyzed separately where field availability differs.

In [206]:
import os
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from dotenv import load_dotenv

load_dotenv(override=True)
dataset_path = os.getenv("LOCAL_DATASET_PATH")
con = duckdb.connect(database=":memory:")

con.execute("""
    CREATE TEMP TABLE contracts AS
    SELECT * FROM read_csv_auto(
        '{0}', delim=',', header=true, strict_mode=false, all_varchar=true, parallel=false
    )
""".format(dataset_path))

# Analysis view: excludes Defence, casts types, filters to valid reporting periods
con.execute("""
    CREATE TEMP VIEW analysis AS
    SELECT *,
        TRY_CAST(contract_value AS DOUBLE) AS val,
        TRY_CAST(original_value AS DOUBLE) AS orig_val,
        TRY_CAST(amendment_value AS DOUBLE) AS amend_val,
        LEFT(reporting_period, 9) AS fiscal_year,
        RIGHT(reporting_period, 2) AS quarter,
        CASE 
            WHEN reporting_period < '2019-2020' THEN 'Pre-2019'
            WHEN reporting_period >= '2019-2020' AND reporting_period < '2022-2023' THEN '2019-2022'
            WHEN reporting_period >= '2022-2023' THEN 'Post-2022'
        END AS era
    FROM contracts
    WHERE owner_org_title NOT LIKE 'National Defence%'
      AND reporting_period LIKE '____-____-Q_'
""")

con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT procurement_id) AS unique_ids FROM analysis").pl()

rows,unique_ids
i64,i64
814070,657374


## Data completeness

| Scope | Reporting | Impact |
|---|---|---|
| Pre-2019 | Voluntary. `reporting_period` ~22% missing. Process fields 35-97% missing. | Volume counts inflated by selective disclosure. Used as supporting evidence only. |
| Post-2019 | Mandatory. Core fields reliable. | Primary evidence base for this analysis. |
| All Years | Combined | Shows long-term patterns. Volume metrics influenced by pre-2019 bias. |

In [207]:
# Era breakdown
con.sql("""
    SELECT era,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='C' THEN 1 ELSE 0 END) AS contracts,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='C' THEN val ELSE 0 END)/1e9, 2) AS contract_val_B
    FROM analysis
    GROUP BY era ORDER BY era
""").pl()

era,total_rows,contracts,amendments,contract_val_B
str,i64,"decimal[38,0]","decimal[38,0]",f64
"""2019-2022""",197472,145372,48926,31.44
"""Post-2022""",252771,187645,62335,44.07
"""Pre-2019""",363827,218188,58920,28.45


### Data quality note: vendor name inconsistency

Same vendor appears under different spellings. Our insights do not depend on exact vendor matching, so we proceed as-is.

In [208]:
# Duplicate vendor names
con.sql("""
    SELECT a.vendor_name AS spelling_1, b.vendor_name AS spelling_2,
           a.cnt AS count_1, b.cnt AS count_2
    FROM (SELECT vendor_name, COUNT(*) AS cnt FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) a
    JOIN (SELECT vendor_name, COUNT(*) AS cnt FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name HAVING COUNT(*)>50) b
    ON UPPER(REPLACE(REPLACE(a.vendor_name,'.',''),' ','')) = UPPER(REPLACE(REPLACE(b.vendor_name,'.',''),' ',''))
    AND a.vendor_name < b.vendor_name
    ORDER BY a.cnt + b.cnt DESC
    LIMIT 10
""").pl()

spelling_1,spelling_2,count_1,count_2
str,str,i64,i64
"""CANADIAN CORPS OF COMMISSIONAI…","""Canadian Corps of Commissionai…",5032,699
"""VERITAAQ TECHNOLOGY HOUSE INC.""","""Veritaaq Technology House Inc.""",923,3091
"""VERITAAQ TECHNOLOGY HOUSE INC""","""Veritaaq Technology House Inc.""",305,3091
"""NISHA TECHNOLOGIES INC.""","""Nisha Technologies Inc.""",2248,741
"""STANTEC CONSULTING LTD""","""STANTEC CONSULTING LTD.""",597,2319
"""NISHA TECHNOLOGIES INC""","""NISHA TECHNOLOGIES INC.""",574,2248
"""STANTEC CONSULTING LTD.""","""Stantec Consulting Ltd.""",2319,455
"""S.I. SYSTEMS""","""S.i. Systems""",1987,611
"""SOFTCHOICE CORPORATION""","""Softchoice Corporation""",1891,543


---

## Insight 1: Q4 spending surge

Canada's fiscal year ends March 31. If departments rush to spend remaining budget, Q4 (Jan-Mar) should show higher contract counts and values. We compare three scopes throughout: pre-2019 (voluntary reporting), post-2019 (mandatory reporting), and all years.

In [209]:
# Q4 pattern by scope: pre-2019, post-2019, all years (contracts only)
q_scopes = con.sql("""
    SELECT scope, quarter, COUNT(*) AS contracts, ROUND(AVG(val), 0) AS avg_value
    FROM (
        SELECT *, 'Pre-2019' AS scope FROM analysis WHERE instrument_type='C' AND reporting_period < '2019-2020'
        UNION ALL
        SELECT *, 'Post-2019' AS scope FROM analysis WHERE instrument_type='C' AND reporting_period >= '2019-2020'
        UNION ALL
        SELECT *, 'All Years' AS scope FROM analysis WHERE instrument_type='C'
    ) combined
    GROUP BY scope, quarter ORDER BY scope, quarter
""").pl()
q_scopes

scope,quarter,contracts,avg_value
str,str,i64,f64
"""All Years""","""Q1""",125545,189701.0
"""All Years""","""Q2""",120994,168310.0
"""All Years""","""Q3""",136075,173164.0
"""All Years""","""Q4""",168591,214868.0
"""Post-2019""","""Q1""",77861,226566.0
…,…,…,…
"""Post-2019""","""Q4""",91449,292705.0
"""Pre-2019""","""Q1""",47684,129507.0
"""Pre-2019""","""Q2""",45129,145911.0


In [210]:
# Q4 avg contract value: pre-2019 vs post-2019 vs all years
colors = ['#4878A8', '#4878A8', '#4878A8', '#D04040']
scopes = ['Pre-2019', 'Post-2019', 'All Years']

fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{s}" for s in scopes], shared_yaxes=True)

for i, scope in enumerate(scopes):
    d = q_scopes.filter(pl.col("scope") == scope)
    fig.add_trace(go.Bar(x=d["quarter"].to_list(), y=d["avg_value"].to_list(),
        marker_color=colors, text=[f'${v/1000:.0f}K' for v in d["avg_value"].to_list()],
        textposition='outside', cliponaxis=False, showlegend=False), row=1, col=i+1)

fig.update_yaxes(title_text="Avg Contract Value ($)", row=1, col=1)
fig.update_layout(title_text="Q4 avg contract value by scope (new contracts only)",
    height=450, margin=dict(t=60))
fig.show()

Pre-2019 shows no Q4 value premium (voluntary reporting may skew this). Post-2019 shows a clear Q4 premium, intensifying in recent years.

**Note**: `commodity_type` and `instrument_type` were not mandatory before 2019, so the pre-2019 panels in commodity and transaction type charts are based on incomplete data. Post-2019 is the reliable comparison.

In [211]:
# Q4 share comparison: pre-2019, post-2019, overall
con.sql("""
    SELECT 
        scope,
        COUNT(*) AS total_contracts,
        SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END) AS q4_contracts,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct_count,
        ROUND(SUM(CASE WHEN quarter='Q4' THEN val ELSE 0 END)*100.0/SUM(val), 1) AS q4_pct_value,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 2) AS q4_multiplier
    FROM (
        SELECT *, 'Pre-2019' AS scope FROM analysis WHERE instrument_type='C' AND reporting_period < '2019-2020'
        UNION ALL
        SELECT *, 'Post-2019' AS scope FROM analysis WHERE instrument_type='C' AND reporting_period >= '2019-2020'
        UNION ALL
        SELECT *, 'All Years' AS scope FROM analysis WHERE instrument_type='C'
    ) combined
    GROUP BY scope
    ORDER BY scope
""").pl()

scope,total_contracts,q4_contracts,q4_pct_count,q4_pct_value,q4_avg,other_avg,q4_multiplier
str,i64,"decimal[38,0]",f64,f64,f64,f64,f64
"""All Years""",551205,168591,30.6,34.8,214868.0,177056.0,1.21
"""Post-2019""",333017,91449,27.5,35.4,292705.0,201787.0,1.45
"""Pre-2019""",218188,77142,35.4,33.2,122576.0,134695.0,0.91


The table confirms: pre-2019 Q4 had higher volume (35.4% of contracts) but lower value (0.91x multiplier) - consistent with voluntary bulk reporting. Post-2019 shows the real value pattern: Q4 contracts are worth more, not just more numerous. The multiplier grows from 1.45x (post-2019 overall) and is strongest in the most recent data.

In [ ]:
# Department Q4 multipliers: pre-2019 vs post-2019 vs all years
fig = make_subplots(rows=1, cols=3, subplot_titles=['Pre-2019', 'Post-2019', 'All Years'])

for i, (scope, filt) in enumerate([
    ('Pre-2019', "reporting_period < '2019-2020'"),
    ('Post-2019', "reporting_period >= '2019-2020'"),
    ('All Years', '1=1')
]):
    dept = con.sql(f"""
        SELECT LEFT(SPLIT_PART(owner_org_title, ' | ', 1), 30) AS department,
            ROUND(AVG(CASE WHEN quarter='Q4' THEN val END) / 
                NULLIF(AVG(CASE WHEN quarter!='Q4' THEN val END), 0), 1) AS q4_multiplier
        FROM analysis WHERE instrument_type = 'C' AND {filt}
        GROUP BY owner_org_title HAVING COUNT(*) >= 200
        ORDER BY q4_multiplier DESC LIMIT 5
    """).pl().to_pandas()
    
    fig.add_trace(go.Bar(y=dept['department'], x=dept['q4_multiplier'], orientation='h',
        marker_color=['#D04040' if v > 2 else '#E8A040' if v > 1.5 else '#4878A8' for v in dept['q4_multiplier']],
        text=[f"{v}x" for v in dept['q4_multiplier']], textposition='outside', cliponaxis=False,
        showlegend=False), row=1, col=i+1)

fig.add_vline(x=1.0, line_dash="dash", line_color="gray")
fig.update_layout(title="Top departments by Q4 value multiplier",
    height=400, margin=dict(t=60, l=10))
for j in range(1, 4):
    fig.update_yaxes(autorange="reversed", row=1, col=j)
fig.update_xaxes(title_text="Q4 multiplier", row=1, col=2)
fig.show()

In [214]:
# Non-competitive rate: Q4 vs other quarters (post-2019)
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4 (Jan-Mar)' ELSE 'Q1-Q3 (Apr-Dec)' END AS period,
        COUNT(*) AS total_contracts,
        SUM(CASE WHEN solicitation_procedure='TN' THEN 1 ELSE 0 END) AS non_competitive,
        ROUND(SUM(CASE WHEN solicitation_procedure='TN' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS non_comp_pct
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

period,total_contracts,non_competitive,non_comp_pct
str,i64,"decimal[38,0]",f64
"""Q1-Q3 (Apr-Dec)""",241568,94357,39.1
"""Q4 (Jan-Mar)""",91449,37774,41.3


In [215]:
# Q4 value surge by commodity type: pre-2019 vs post-2019 vs all years
comm_scopes = {}
for scope, filt in [('Pre-2019', "reporting_period < '2019-2020'"), 
                     ('Post-2019', "reporting_period >= '2019-2020'"),
                     ('All Years', '1=1')]:
    comm_scopes[scope] = con.sql(f"""
        SELECT CASE commodity_type WHEN 'S' THEN 'Services' WHEN 'G' THEN 'Goods' WHEN 'C' THEN 'Construction' END AS commodity,
            ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg,
            ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg
        FROM analysis WHERE instrument_type = 'C' AND commodity_type IN ('S','G','C') AND {filt}
        GROUP BY commodity_type ORDER BY commodity_type
    """).pl().to_pandas()

fig = make_subplots(rows=1, cols=3, subplot_titles=['Pre-2019', 'Post-2019', 'All Years'], shared_yaxes=True)

for i, scope in enumerate(['Pre-2019', 'Post-2019', 'All Years']):
    df = comm_scopes[scope]
    fig.add_trace(go.Bar(name='Q1-Q3', x=df['commodity'], y=df['other_avg'], marker_color='#4878A8',
        text=[f'${v/1000:.0f}K' for v in df['other_avg']], textposition='outside', cliponaxis=False,
        showlegend=(i==0)), row=1, col=i+1)
    fig.add_trace(go.Bar(name='Q4', x=df['commodity'], y=df['q4_avg'], marker_color='#D04040',
        text=[f'${v/1000:.0f}K' for v in df['q4_avg']], textposition='outside', cliponaxis=False,
        showlegend=(i==0)), row=1, col=i+1)

fig.update_layout(barmode='group', title="Q4 construction surge across scopes",
    height=500, margin=dict(t=60))
fig.update_yaxes(title_text="Avg Contract Value ($)", row=1, col=1)
fig.show()

In [217]:
# Q4 concentration by instrument type: pre-2019 vs post-2019 vs all years
fig = make_subplots(rows=1, cols=3, subplot_titles=['Pre-2019', 'Post-2019', 'All Years'], shared_yaxes=True)

for i, (scope, filt) in enumerate([
    ('Pre-2019', "reporting_period < '2019-2020'"),
    ('Post-2019', "reporting_period >= '2019-2020'"),
    ('All Years', '1=1')
]):
    inst = con.sql(f"""
        SELECT CASE instrument_type WHEN 'C' THEN 'New Contracts' WHEN 'A' THEN 'Amendments' WHEN 'SOSA' THEN 'SOSAs' END AS type,
            ROUND(SUM(CASE WHEN quarter='Q4' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS q4_pct
        FROM analysis WHERE instrument_type IS NOT NULL AND {filt}
        GROUP BY instrument_type ORDER BY q4_pct DESC
    """).pl().to_pandas()
    
    fig.add_trace(go.Bar(x=inst['type'], y=inst['q4_pct'],
        marker_color=[('#D04040' if p > 30 else '#E8A040' if p > 25 else '#4878A8') for p in inst['q4_pct']],
        text=[f"{v}%" for v in inst['q4_pct']], textposition='outside', cliponaxis=False,
        showlegend=False), row=1, col=i+1)

fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="Expected 25%")
fig.update_layout(title="Q4 share by transaction type across scopes",
    height=450, margin=dict(t=60))
fig.update_yaxes(title_text="% in Q4", row=1, col=1)
fig.show()

Two channels drive the Q4 rush across all scopes: new contracts and amendments. Amendments are consistently more concentrated in Q4 than new contracts, regardless of era. The sole-source rate is also slightly higher in Q4 (post-2019 where the field is available).

In [218]:
# Year-over-year Q4 trend
q4_trend = con.sql("""
    SELECT fiscal_year,
        ROUND(AVG(CASE WHEN quarter='Q4' THEN val END), 0) AS q4_avg_val,
        ROUND(AVG(CASE WHEN quarter!='Q4' THEN val END), 0) AS other_avg_val
    FROM analysis
    WHERE instrument_type = 'C' AND reporting_period >= '2019-2020' AND fiscal_year <= '2024-2025'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl().to_pandas()

fig = go.Figure()
fig.add_trace(go.Scatter(x=q4_trend['fiscal_year'], y=q4_trend['other_avg_val'], mode='lines+markers',
    name='Q1-Q3 avg value', line=dict(color='#4878A8', width=2)))
fig.add_trace(go.Scatter(x=q4_trend['fiscal_year'], y=q4_trend['q4_avg_val'], mode='lines+markers',
    name='Q4 avg value', line=dict(color='#D04040', width=3)))
fig.update_layout(title="Q4 vs Q1-Q3 average contract value by fiscal year", yaxis_title="Average Contract Value ($)",
    xaxis_title="Fiscal Year", height=400, hovermode='x unified')
fig.show()

### Insight 1 summary

*Volume measured across all transaction types. Value measured on new contracts only (amendment values are cumulative totals, not new spending).*

**Volume**: Q4 sees +20% more procurement activity than Q1-Q3 average post-2019 (mandatory reporting), +38% across all years (inflated by voluntary pre-2019 reporting). New contracts alone show +32%.

**Value**: 34.8% of total contract value lands in Q4 all years, 35.4% post-2019 (expected: 25%). Q4 average is $215K vs $177K. Post-2022, the multiplier is 1.89x. Construction: 3.4x all years, 5.84x post-2019. Q4 dollar premium post-2019: $8.31B.

**Two channels**: New contracts (27.5% in Q4) and amendments (32.7% in Q4) both drive the surge. Sole-source rate is higher in Q4 (41.3% vs 39.1%).

**Recommendation**: Flag high-value Q4 construction contracts. Track Q4 sole-source and amendment rates by department. Consider multi-year budgeting.

**Caveat**: `reporting_period` is when the contract was *reported*, not *awarded*. Only mandatory after 2019-01-01.

---

## Insight 2: Amendment growth

Amendment rate jumped from 16% to 25% since 2019, and some contracts grow far beyond their original scope.

In [219]:
# Amendment rate by era
con.sql("""
    SELECT 
        COALESCE(era, 'Overall') AS period,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='C' THEN 1 ELSE 0 END) AS contracts,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amendment_rate_pct
    FROM analysis
    GROUP BY ROLLUP(era)
    ORDER BY CASE WHEN era IS NULL THEN 'ZZZ' ELSE era END
""").pl()

period,total_rows,contracts,amendments,amendment_rate_pct
str,i64,"decimal[38,0]","decimal[38,0]",f64
"""2019-2022""",197472,145372,48926,24.8
"""Post-2022""",252771,187645,62335,24.7
"""Pre-2019""",363827,218188,58920,16.2
"""Overall""",814070,551205,170181,20.9


In [220]:
# Amendment rate by fiscal year
amend_by_year = con.sql("""
    SELECT fiscal_year, ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_pct
    FROM analysis WHERE fiscal_year >= '2017-2018' AND fiscal_year <= '2024-2025' AND fiscal_year != '2018-2018'
    GROUP BY fiscal_year ORDER BY fiscal_year
""").pl().to_pandas()

fig = px.bar(amend_by_year, x='fiscal_year', y='amend_pct', text='amend_pct',
    title="Amendment rate by fiscal year (excl. Defence)", color_discrete_sequence=['#4878A8'])
fig.add_hline(y=20, line_dash="dash", line_color="gray", annotation_text="20% baseline")
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(xaxis_title="Fiscal Year", yaxis_title="Amendment rate (%)", height=500)
fig.show()

Overall amendment rate is ~21%, masking a clear shift: 16.2% pre-2019, then ~25% and stable. The rate alone does not reveal whether these are small extensions or major scope changes.

In [221]:
# Distribution of contract growth via amendments
growth = con.sql("""
    SELECT 
        CASE 
            WHEN growth_pct < 0 THEN 'Decreased'
            WHEN growth_pct = 0 THEN 'No change'
            WHEN growth_pct <= 25 THEN '1-25%'
            WHEN growth_pct <= 50 THEN '26-50%'
            WHEN growth_pct <= 100 THEN '51-100%'
            WHEN growth_pct <= 500 THEN '101-500%'
            ELSE '500%+'
        END AS growth_bucket,
        COUNT(*) AS contracts,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct
    FROM (
        SELECT procurement_id,
            ROUND((TRY_CAST(MAX(contract_value) AS DOUBLE) / 
                   NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100, 0) AS growth_pct
        FROM analysis
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1 AND COUNT(DISTINCT instrument_type) > 1
    ) sub
    WHERE growth_pct IS NOT NULL
    GROUP BY growth_bucket
    ORDER BY MIN(growth_pct)
""").pl()
growth

growth_bucket,contracts,pct
str,i64,f64
"""Decreased""",4294,6.2
"""No change""",18727,26.9
"""1-25%""",11225,16.1
"""26-50%""",7892,11.3
"""51-100%""",10915,15.7
"""101-500%""",14818,21.3
"""500%+""",1806,2.6


In [222]:
# Amendment growth distribution
color_map = {'Decreased': '#6BAF6B', 'No change': '#888888', '1-25%': '#4878A8', '26-50%': '#4878A8',
             '51-100%': '#4878A8', '101-500%': '#E8A040', '500%+': '#D04040'}
growth_pd = growth.to_pandas()
growth_pd['color'] = growth_pd['growth_bucket'].map(color_map)

fig = px.bar(growth_pd, y='growth_bucket', x='pct', orientation='h', text='pct',
    title="How much do amendments grow contract values?", color='growth_bucket',
    color_discrete_map=color_map)
fig.update_traces(texttemplate='%{text}% (%{customdata[0]:,})', textposition='outside',
    customdata=growth_pd[['contracts']].values)
fig.update_layout(yaxis=dict(autorange="reversed", title=""), xaxis_title="% of amended contracts",
    height=400, showlegend=False)
fig.show()

~24% of amended contracts more than double in value. Checking the most extreme cases and which commodity types are hit hardest.

In [223]:
# Top 10 contracts by absolute growth (original > $100K)
con.sql("""
    SELECT 
        procurement_id,
        vendor_name,
        SPLIT_PART(owner_org_title, ' | ', 1) AS department,
        amendment_count,
        ROUND(original_M, 1) AS original_M,
        ROUND(final_M, 1) AS final_M,
        ROUND(growth_pct, 0) AS growth_pct
    FROM (
        SELECT procurement_id,
            MAX(vendor_name) AS vendor_name,
            MAX(owner_org_title) AS owner_org_title,
            COUNT(*) - 1 AS amendment_count,
            TRY_CAST(MIN(original_value) AS DOUBLE)/1e6 AS original_M,
            TRY_CAST(MAX(contract_value) AS DOUBLE)/1e6 AS final_M,
            (TRY_CAST(MAX(contract_value) AS DOUBLE) / 
             NULLIF(TRY_CAST(MIN(original_value) AS DOUBLE), 0) - 1) * 100 AS growth_pct
        FROM analysis
        WHERE procurement_id IS NOT NULL
        GROUP BY procurement_id
        HAVING COUNT(*) > 1
    ) sub
    WHERE original_M > 0.1 AND growth_pct > 100
    ORDER BY final_M - original_M DESC
    LIMIT 10
""").pl()

procurement_id,vendor_name,department,amendment_count,original_M,final_M,growth_pct
str,str,str,i64,f64,f64,f64
"""E112560004""","""BGIS GLOBAL INTEGRATED SOLUTIO…","""Public Services and Procuremen…",1,768.5,1891.0,146.0
"""700385683""","""BELL MOBILITY INC""","""Shared Services Canada""",16,18.6,985.6,5205.0
"""EW70241166""","""Parsons Inc""","""Public Services and Procuremen…",23,49.8,991.4,1890.0
"""4500065824""","""VANCOUVER SHIPYARDS CO LTD""","""Fisheries and Oceans Canada""",2,179.9,849.9,372.0
"""6D06304342""","""SWITCH HEALTH HOLDINGS INC.""","""Public Health Agency of Canada""",3,100.0,751.6,652.0
"""G746204002""","""D+H CORPORATION""","""Employment and Social Developm…",14,270.7,897.0,231.0
"""EH90090297""","""Ellisdon Corporation""","""Public Services and Procuremen…",22,0.2,623.4,263069.0
"""EP75813490""","""PCL CONSTRUCTORS CANADA INC.""","""Public Services and Procuremen…",10,360.7,970.3,169.0
"""EP74851887""","""WSP CANADA INC""","""Public Services and Procuremen…",8,127.4,689.9,442.0


In [224]:
# Amendment rate by commodity type (post-2019)
amend_commodity = con.sql("""
    SELECT commodity_type,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020' AND commodity_type IS NOT NULL
    GROUP BY commodity_type ORDER BY amend_rate_pct DESC
""").pl()
amend_commodity

commodity_type,total_rows,amendments,amend_rate_pct
str,i64,"decimal[38,0]",f64
"""S""",282226,88669,31.4
"""C""",35016,10652,30.4
"""G""",132990,11929,9.0


In [225]:
# Amendment rate by commodity type
fig = px.bar(amend_commodity.to_pandas(), x='commodity_type', y='amend_rate_pct', text='amend_rate_pct',
    title="Services and construction are amended 3x more than goods",
    color='commodity_type', color_discrete_map={'S': '#D04040', 'C': '#E8A040', 'G': '#4878A8'},
    labels={'commodity_type': 'Commodity Type', 'amend_rate_pct': 'Amendment Rate (%)'})
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.update_layout(height=500, showlegend=False)
fig.show()

In [226]:
# Amendment rate: Q4 vs other quarters (post-2019)
con.sql("""
    SELECT 
        CASE WHEN quarter = 'Q4' THEN 'Q4 (Jan-Mar)' ELSE 'Q1-Q3 (Apr-Dec)' END AS period,
        COUNT(*) AS total_rows,
        SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END) AS amendments,
        ROUND(SUM(CASE WHEN instrument_type='A' THEN 1 ELSE 0 END)*100.0/COUNT(*), 1) AS amend_rate_pct
    FROM analysis
    WHERE reporting_period >= '2019-2020'
    GROUP BY period ORDER BY period
""").pl()

period,total_rows,amendments,amend_rate_pct
str,i64,"decimal[38,0]",f64
"""Q1-Q3 (Apr-Dec)""",321263,74925,23.3
"""Q4 (Jan-Mar)""",128980,36336,28.2


In [227]:
# Q4 vs other amendment rate
fig = go.Figure(go.Bar(x=['Q1-Q3', 'Q4'], y=[23.3, 28.2], marker_color=['#4878A8', '#D04040'],
    text=['23.3%', '28.2%'], textposition='outside', cliponaxis=False))
fig.add_hline(y=25, line_dash="dash", line_color="gray", annotation_text="25%")
fig.update_layout(title="Amendments spike in Q4", yaxis_title="Amendment rate (%)", height=350)
fig.show()

### Insight 2 summary

**Pattern**: 1 in 4 rows is an amendment (up from 1 in 6 pre-2019). Services at 31.8%, construction at 30.4%, goods at 9%. Amendments spike in Q4 (28.2% vs 23.3%).

**Evidence**: Bell Mobility grew from $18.6M to $985.6M (16 amendments). Parsons Inc grew from $49.8M to $991.4M (23 amendments). These effectively bypassed the competitive process.

**Recommendation**: Set amendment thresholds -- if cumulative amendments exceed 50% of original value, require justification or re-compete. Track amendment rates by commodity type. Watch Q4 amendments specifically.

**Caveat**: Growth estimated by comparing `MAX(contract_value)` to `MIN(original_value)` across rows sharing a `procurement_id`.

---

## Insight 3: Vendor concentration amplifies amendment risk

Top 50 vendors hold over half of all spend, and their amendment rate is nearly double the rest.

In [228]:
# Vendor concentration -- top 10/50/100 share of total spend
conc = con.sql("""
    WITH v AS (
        SELECT vendor_name, SUM(val) as total_val,
               ROW_NUMBER() OVER (ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL
        GROUP BY vendor_name
    ),
    gt AS (SELECT SUM(total_val) as g, COUNT(*) as total_vendors FROM v)
    SELECT 
        (SELECT total_vendors FROM gt) as total_vendors,
        (SELECT ROUND(g/1e9, 1) FROM gt) as total_B,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 10) * 100.0 / (SELECT g FROM gt), 1) as top10_pct,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 50) * 100.0 / (SELECT g FROM gt), 1) as top50_pct,
        ROUND((SELECT SUM(total_val) FROM v WHERE rn <= 100) * 100.0 / (SELECT g FROM gt), 1) as top100_pct
""").pl()
conc

total_vendors,total_B,top10_pct,top50_pct,top100_pct
i64,f64,f64,f64,f64
126469,669.9,28.7,55.0,62.6


In [229]:
# Do top vendors get amended more?
tier_amend = con.sql("""
    WITH v AS (
        SELECT vendor_name, SUM(val) as total_val,
               ROW_NUMBER() OVER (ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL GROUP BY vendor_name
    )
    SELECT 
        CASE WHEN v.rn <= 50 THEN 'Top 50 vendors' ELSE 'All others' END as tier,
        COUNT(*) as rows,
        ROUND(100.0 * SUM(CASE WHEN a.instrument_type='A' THEN 1 ELSE 0 END) / COUNT(*), 1) as amend_rate
    FROM analysis a
    JOIN v ON a.vendor_name = v.vendor_name
    GROUP BY CASE WHEN v.rn <= 50 THEN 'Top 50 vendors' ELSE 'All others' END
    ORDER BY MIN(v.rn)
""").pl()

fig = go.Figure(go.Bar(
    x=tier_amend["tier"].to_list(), y=tier_amend["amend_rate"].to_list(),
    marker_color=['#D04040', '#4878A8'],
    text=[f"{v}%" for v in tier_amend["amend_rate"].to_list()],
    textposition="outside", cliponaxis=False))
fig.update_layout(title="Top vendors have nearly double the amendment rate",
    yaxis_title="Amendment rate (%)", template="plotly_white", height=400, margin=dict(t=60))
fig.show()

In [230]:
# Department dependency -- which rely most on a single vendor?
con.sql("""
    WITH dept_vendor AS (
        SELECT owner_org_title, vendor_name, SUM(val) as vendor_total,
               SUM(SUM(val)) OVER (PARTITION BY owner_org_title) as dept_total,
               ROW_NUMBER() OVER (PARTITION BY owner_org_title ORDER BY SUM(val) DESC) as rn
        FROM analysis WHERE vendor_name IS NOT NULL AND val > 0
        GROUP BY owner_org_title, vendor_name
    )
    SELECT SPLIT_PART(owner_org_title, ' | ', 1) as department,
           vendor_name as top_vendor,
           ROUND(dept_total/1e9, 2) as dept_total_B,
           ROUND(vendor_total/1e9, 2) as vendor_total_B,
           ROUND(100.0 * vendor_total / dept_total, 1) as dependency_pct
    FROM dept_vendor
    WHERE rn = 1 AND dept_total > 1e9
    ORDER BY dependency_pct DESC
    LIMIT 10
""").pl()

department,top_vendor,dept_total_B,vendor_total_B,dependency_pct
str,str,f64,f64,f64
"""Department of Housing, Infrast…","""GROUPE SIGNATURE SUR LE""",25.05,10.06,40.1
"""Canadian Space Agency""","""MDA SYSTEMS LTD.""",40.29,15.7,39.0
"""Fisheries and Oceans Canada""","""VANCOUVER SHIPYARDS CO LTD""",63.51,24.52,38.6
"""Treasury Board of Canada Secre…","""Sun Life Assurance Company of …",5.71,2.19,38.3
"""Canada Border Services Agency""","""Deloitte Inc""",14.02,4.36,31.1
"""Employment and Social Developm…","""D+H CORPORATION""",34.93,10.05,28.8
"""Statistics Canada""","""MICROSOFT CANADA INC.""",1.2,0.33,27.7
"""Health Canada""","""SC2.0 STEPPED CARE SOLUTIONS I…",6.28,1.57,25.0
"""Elections Canada""","""KYNDRYL CANADA LTEE""",2.51,0.6,23.8


### Insight 3 summary

**Pattern**: Top 50 vendors (out of 126,000+) hold ~55% of all spending. Their amendment rate is ~38% -- nearly double the ~21% for others. 18 of 97 departments have >30% of spend with a single vendor.

**Why it matters**: Vendor concentration creates dependency risk and reduces negotiating leverage. The highest-spend vendors are also the ones whose contracts change the most after award, connecting directly to Insight 2.

**Recommendation**: Map vendor dependency for critical services. Diversify the vendor base for recurring categories. Monitor whether top-vendor contracts grow disproportionately through amendments.

---

## The cycle

These three insights form a self-reinforcing loop: year-end budget pressure creates rushed awards (Insight 1). Those contracts grow through amendments (Insight 2), disproportionately benefiting established vendors (Insight 3), who then receive even more spend. The competitive process used for the original award becomes less meaningful over time.

## Federal benchmarks

| Metric | All Years | Post-2019 | Scope |
|---|---|---|---|
| Q4 volume surge | +38% vs Q1-Q3 avg | +20% vs Q1-Q3 avg | All transaction types |
| Q4 share of contract value | 34.8% (expected: 25%) | 35.4% | Contracts only |
| Q4 avg contract value vs Q1-Q3 | $215K vs $177K | -- | Contracts only |
| Q4 construction multiplier | 3.4x | 5.84x | Contracts only |
| Q4 dollar premium | -- | $8.31B | Contracts only |
| Q4 sole-source rate vs Q1-Q3 | -- | 41.3% vs 39.1% | Contracts only, post-2019 |
| Q4 share of new contracts | -- | 27.5% | Post-2019 |
| Q4 share of amendments | -- | 32.7% | Post-2019 |
| Amendment rate | -- | ~25% | Post-2019 |
| Services amendment rate | -- | 31.4% | Post-2019 |
| Top 50 vendor share of spend | ~55% | -- | All eras |
| Top 50 vendor amendment rate | ~38% (vs ~21% others) | -- | All eras |
| Depts with >30% single-vendor dependency | 18 of 97 | -- | All eras |

## Next steps

1. Do contracts *awarded* in Q4 get amended more than Q1-Q3 contracts later on?
2. Which contract descriptions drive the Q4 volume surge (temporary help, consulting)?
3. Vendor name normalization - fuzzy matching would reveal true concentration.
4. Do top vendors bid low and grow through amendments?